In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set visualization settings
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## Load the dataset

In [ ]:
df = pd.read_csv('Loan_Default.csv')
print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape}")

* We have a total of 34 features of 1,48,670 rows

# Features :

- **ID** = Unique identifier for each loan record.

- **year** = Year in which the loan was issued.

- **loan_limit** = Indicates whether the loan exceeds a predefined lending limit.

- **Gender** = Gender of the primary loan applicant.

- **approv_in_adv** = Whether the loan was approved in advance (pre-approved) or not.

- **loan_type** = Type/category of the loan (e.g., conventional, government-backed).

- **loan_purpose** = Purpose for which the loan was taken (e.g., purchase, refinance).

- **Credit_Worthiness** = Assessment category of the applicant’s credit quality.

- **open_credit** = Indicates if the applicant currently has open lines of credit.

- **business_or_commercial** = Whether the loan is for business/commercial use.

- **loan_amount** = Total amount borrowed.

- **rate_of_interest** = Interest rate applied to the loan.

- **Interest_rate_spread** = Difference between the loan’s interest rate and a benchmark rate.

- **Upfront_charges** = Initial fees charged at the start of the loan.

- **term** = Loan duration in months.

- **Neg_ammortization** = Whether the loan allows negative amortization (unpaid interest added to principal).

- **interest_only** = Indicates if the loan requires only interest payments for a certain period.

- **lump_sum_payment** = Whether a large one-time payment option exists.

- **property_value** = Estimated value of the property tied to the loan.

- **construction_type** = Type of property construction (e.g., site-built, manufactured).

- **occupancy_type** = Whether the property is owner-occupied, rented, etc.

- **Secured_by** = Type of asset securing the loan.

- **total_units** = Number of housing units in the property.

- **income** = Annual income of the applicant.

- **credit_type** = Type of credit evaluation used (e.g., traditional, alternative).

- **Credit_Score** = Numerical credit score of the applicant.

- **co-applicant_credit_type** = Credit type of the co-applicant (if any).

- **age** = Age of the primary applicant.

- **submission_of_application** = Method used to submit the loan application.

- **LTV** = Loan-to-Value ratio (loan amount divided by property value).

- **Region** = Geographic region of the property or applicant.

- **Security_Type** = Type of security instrument used for the loan.

- **Status** = Loan outcome (e.g., defaulted or non-defaulted).

- **dtir1** = Debt-to-Income ratio of the applicant.


## Display basic information about dataset

In [ ]:
print("Dataset Info:")
print(df.info())


* In our dataset, some features are in int variables and some categorical variables

In [ ]:
print("Missing Values:")
print(df.isnull().sum())

- There are null values that must be filled

In [ ]:
# Display first few rows and statistical summary
print("First few rows:")
print(df.head())


In [ ]:
print("\nStatistical Summary:")
print(df.describe())

In [ ]:
df['Status'].value_counts()

- loan defaulted (1) = 36,639
- Not defaulted (0) = 1,12,031  

In [ ]:
df.duplicated().sum()

In [ ]:
msno.matrix(df)
plt.show()

## Checking if loan status is dependant on gender

In [ ]:
fig,ax=plt.subplots()
sns.barplot(data=df,x='Status',y='income',hue='Gender')
fig.set_size_inches([12,6])
plt.show()

In [ ]:
df['Gender'].value_counts()

## How loan defaulter is dependant to credit score and loan amount

In [ ]:
fig, ax=plt.subplots()
sns.scatterplot(data=df,x='Credit_Score', y='loan_amount', hue='Status')
fig.set_size_inches([12,6])
plt.show()

In [ ]:
sns.countplot(x='Status', data=df)
plt.show()


- The dataset exhibits class imbalance, with a significantly higher proportion of non-default loans compared to default loans. This imbalance may cause the model to become biased toward predicting the majority class.

In [ ]:
sns.countplot(data=df, x='age')

In [ ]:
sns.boxplot(data=df, x="Status", y="rate_of_interest")


- The boxplot shows that defaulters tend to have slightly higher median interest rates compared to non-defaulters. However, significant overlap between the distributions indicates that interest rate alone is not sufficient to predict loan default.

In [ ]:
sns.scatterplot(
    data=df,
    x="rate_of_interest",
    y="loan_amount",
    hue="Status",
    alpha=0.5   # helps reduce overlap
)
plt.title("Loan Amount vs. Rate of Interest by Loan Status")
plt.show()

- The scatter plot reveals significant overlap between default and non-default loans across interest rate and loan amount, indicating that these variables alone do not provide strong class separation.

In [ ]:
plt.figure(figsize=(20,20))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm')
plt.show()


- The correlation heatmap indicates strong relationships among financial variables such as loan amount and property value (r=0.73). However, no single variable exhibits strong correlation with loan default status, suggesting that default prediction requires multi-feature modeling rather than reliance on individual predictors.

In [ ]:
df = df.drop(columns=['year'])


- The year column contained a constant value (2019) across all records and was removed as it provides no predictive information.

In [ ]:
df.head(5)

## Now we will Handle missing values.

In [ ]:
df.isnull().sum()

In [ ]:
#Percentage of missing values from total values.
(df.isnull().mean() * 100)[lambda x : x > 0].sort_values(ascending=False)


In [ ]:
from sklearn.impute import SimpleImputer

# -----------------------------
# 1️⃣ Numerical Columns
# -----------------------------
numeric_cols = [
    'Interest_rate_spread',
    'Upfront_charges',
    'rate_of_interest',
    'term',
    'property_value',
    'income',
    'dtir1',
    'LTV'
]

num_imputer = SimpleImputer(strategy='mean')
df[numeric_cols] = num_imputer.fit_transform(df[numeric_cols])


# -----------------------------
# 2️⃣ Categorical Columns (Including Missing Ones)
# -----------------------------
categorical_cols = [
    'age',
    'loan_limit',
    'loan_purpose',
    'Neg_ammortization',
    'submission_of_application',
    'approv_in_adv'
]

cat_imputer = SimpleImputer(strategy='most_frequent')
df[categorical_cols] = cat_imputer.fit_transform(df[categorical_cols])


In [ ]:
df.isnull().sum()

- Missing numerical values were imputed using mean substitution, while categorical variables were imputed using the most frequent value to preserve distribution characteristics.